<a href="https://colab.research.google.com/github/mithunkumarsr/NeurIPS-MAS-2026/blob/main/Lab_3_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Rigorous Agent Benchmarking
**NeurIPS 2026 Education Track: Multi-Agent Orchestration**

**Author:** Mithun Kumar S R (Google)

We evaluate agents by measuring their internal world model against ground truth. We benchmark against stochastic environments to calculate Token Efficiency ($\eta$) and Statistical Confidence Intervals.

In [1]:
import random
import math

class SyntheticDatabaseEnv:
    def __init__(self, failure_rate=0.2):
        self.failure_rate = failure_rate # 20% chance to simulate a 504 Timeout

    def step(self, action):
        if random.random() < self.failure_rate:
            return {"status": 504, "error": "Gateway Timeout"}
        return {"status": 200, "data": "Success"}

### 1. Testing State Progression (Execution Hallucination)
Does the agent's internal $M_{work}$ accurately reflect the true transition $T(S_t, a_t)$? We use a naive mock agent that assumes all its code executes perfectly.

In [2]:
class MockAgent:
    def predict_next_state(self, action):
        # The flat agent hallucinates that its action always succeeds
        return {"status": 200, "data": "Success"}

agent = MockAgent()
env = SyntheticDatabaseEnv(failure_rate=0.2)

def test_progression(agent, env):
    true_next_state = env.step("query_db")
    predicted_state = agent.predict_next_state("query_db")

    # Catching Execution Hallucination
    return 1 if predicted_state == true_next_state else 0

### 2. Statistical Confidence Intervals
Point estimates (e.g., "85% success rate") are statistically invalid for stochastic systems. We require $N \ge 100$ trials and calculate the 95% Confidence Interval ($CI$).

$$ CI = \hat{p} \pm Z \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} $$

In [3]:
def calculate_confidence_interval(successes: int, n: int, z: float = 1.96):
    p_hat = successes / n
    margin_of_error = z * math.sqrt((p_hat * (1 - p_hat)) / n)
    lower_bound = p_hat - margin_of_error
    upper_bound = p_hat + margin_of_error
    return p_hat, (lower_bound, upper_bound)

print("--- Running ACPBench State Progression Simulator ---")
n_trials = 500
progression_successes = 0

for _ in range(n_trials):
    progression_successes += test_progression(agent, env)

p, ci = calculate_confidence_interval(progression_successes, n_trials)

print(f"Trials Executed: {n_trials}")
print(f"Observed Progression Success Rate: {p:.2%}")
print(f"95% Confidence Interval: [{ci[0]:.2%}, {ci[1]:.2%}]")

if ci[0] < 0.90:
    print("\n[VERDICT] The agent architecture fails the enterprise reliability threshold.")
    print("[FIX] Implement M_ep (Episodic Memory) to mitigate execution hallucinations.")

--- Running ACPBench State Progression Simulator ---
Trials Executed: 500
Observed Progression Success Rate: 80.60%
95% Confidence Interval: [77.13%, 84.07%]

[VERDICT] The agent architecture fails the enterprise reliability threshold.
[FIX] Implement M_ep (Episodic Memory) to mitigate execution hallucinations.
